# Lesson 02: Flow Matching Basics

Flow matching is a simpler alternative to score-based diffusion. Instead of an SDE with stochastic noise, we define a **deterministic ODE** that transports noise to data along straight paths.

In this notebook we will:
1. Implement flow matching with linear interpolation.
2. Train on a 2D toy distribution to build intuition.
3. Visualize the learned velocity field and sampling trajectories.
4. Compare the number of sampling steps needed vs diffusion.

In [ ]:
# Install dependencies
%pip install torch numpy matplotlib --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
import sys
sys.path.insert(0, "src")
from flow_matching import VelocityNet, FlowMatcher

## 1. The Core Idea

| Aspect | Score-based (SDE) | Flow Matching (ODE) |
|--------|-------------------|--------------------|
| Forward | Stochastic noise addition | Linear interpolation |
| Target | Score $\nabla_x \log p$ | Velocity $v = x_1 - x_0$ |
| Sampling | SDE solver (stochastic) | ODE solver (deterministic) |
| Schedule | $\beta_t$ schedule needed | No schedule |
| Steps | ~1000 typical | ~50-100 typical |

## 2. Linear Interpolation

Given noise $x_0 \sim \mathcal{N}(0, I)$ and data $x_1$:

$$x_t = (1-t) \cdot x_0 + t \cdot x_1$$

At $t=0$: pure noise. At $t=1$: pure data. The velocity is constant: $v = x_1 - x_0$.

In [ ]:
# Visualize the interpolation
x_0 = torch.randn(5, 2)  # 5 noise samples
x_1 = torch.tensor([[2., 2.], [-2., 2.], [2., -2.], [-2., -2.], [0., 3.]])

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
times = [0.0, 0.25, 0.5, 0.75, 1.0]

for ax, t_val in zip(axes, times):
    t = torch.full((5,), t_val)
    x_t = (1 - t.unsqueeze(-1)) * x_0 + t.unsqueeze(-1) * x_1
    
    ax.scatter(x_0[:, 0], x_0[:, 1], c='blue', alpha=0.3, label='noise')
    ax.scatter(x_1[:, 0], x_1[:, 1], c='red', alpha=0.3, label='data')
    ax.scatter(x_t[:, 0], x_t[:, 1], c='green', s=80, zorder=5, label='x_t')
    for i in range(5):
        ax.plot([x_0[i, 0], x_1[i, 0]], [x_0[i, 1], x_1[i, 1]], 'k--', alpha=0.2)
    ax.set_title(f't = {t_val}')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')

axes[0].legend(fontsize=8)
plt.suptitle('Linear Interpolation: Noise to Data', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Training Flow Matching on 2D Data

In [ ]:
# Target distribution: 4 Gaussian clusters
def sample_target(n: int) -> torch.Tensor:
    centers = torch.tensor([
        [2.0, 2.0], [-2.0, 2.0], [2.0, -2.0], [-2.0, -2.0]
    ])
    idx = torch.randint(0, 4, (n,))
    return centers[idx] + torch.randn(n, 2) * 0.3

target_samples = sample_target(1000)
plt.figure(figsize=(5, 5))
plt.scatter(target_samples[:, 0], target_samples[:, 1], s=5, alpha=0.5)
plt.title('Target Distribution')
plt.xlim(-4, 4)
plt.ylim(-4, 4)
plt.gca().set_aspect('equal')
plt.show()

In [ ]:
# Create model and trainer
model = VelocityNet(data_dim=2, hidden_dim=256, n_layers=4)
fm = FlowMatcher(model, lr=1e-3)
fm.to(device)

losses = []
for step in range(2000):
    x_1 = sample_target(256).to(device)
    loss = fm.train_step(x_1)
    losses.append(loss)
    if (step + 1) % 500 == 0:
        print(f"Step {step+1}: loss = {loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses)
plt.xlabel('Training step')
plt.ylabel('Flow Matching Loss')
plt.title('Training Loss')
plt.show()

## 4. Sampling via Euler ODE Integration

$$x_{t + \Delta t} = x_t + \Delta t \cdot v_\theta(x_t, t)$$

In [ ]:
samples = fm.sample((1000, 2), n_steps=50, device=device).cpu()

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].scatter(target_samples[:, 0], target_samples[:, 1], s=5, alpha=0.5)
axes[0].set_title('Target Distribution')
axes[0].set_xlim(-4, 4)
axes[0].set_ylim(-4, 4)
axes[0].set_aspect('equal')

axes[1].scatter(samples[:, 0], samples[:, 1], s=5, alpha=0.5, c='orange')
axes[1].set_title('Generated Samples (50 Euler steps)')
axes[1].set_xlim(-4, 4)
axes[1].set_ylim(-4, 4)
axes[1].set_aspect('equal')
plt.tight_layout()
plt.show()

## 5. Visualizing the Velocity Field

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
times = [0.0, 0.3, 0.6, 0.9]

grid_range = np.linspace(-4, 4, 20)
xx, yy = np.meshgrid(grid_range, grid_range)
grid_points = torch.tensor(
    np.stack([xx.flatten(), yy.flatten()], axis=1), dtype=torch.float32
).to(device)

model.eval()
for ax, t_val in zip(axes, times):
    t = torch.full((grid_points.shape[0],), t_val, device=device)
    with torch.no_grad():
        v = model(grid_points, t).cpu().numpy()
    ax.quiver(xx.flatten(), yy.flatten(), v[:, 0], v[:, 1], alpha=0.6)
    ax.scatter(target_samples[:200, 0], target_samples[:200, 1], s=3, c='red', alpha=0.3)
    ax.set_title(f't = {t_val}')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')

plt.suptitle('Learned Velocity Field at Different Times', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Sampling Trajectories

In [ ]:
trajectory = fm.sample_trajectory((8, 2), n_steps=100, device=device, save_every=5)

plt.figure(figsize=(6, 6))
plt.scatter(target_samples[:200, 0], target_samples[:200, 1], s=3, c='red', alpha=0.2, label='target')

colors = plt.cm.viridis(np.linspace(0, 1, 8))
for i in range(8):
    xs = [traj[0][i, 0].cpu().item() for traj in trajectory]
    ys = [traj[0][i, 1].cpu().item() for traj in trajectory]
    plt.plot(xs, ys, '-', c=colors[i], alpha=0.7, linewidth=1.5)
    plt.scatter(xs[0], ys[0], c='blue', s=30, zorder=5)
    plt.scatter(xs[-1], ys[-1], c='green', s=30, zorder=5, marker='*')

plt.title('Sampling Trajectories (blue=start, green=end)')
plt.xlim(-4, 4)
plt.ylim(-4, 4)
plt.gca().set_aspect('equal')
plt.show()

## 7. Effect of Number of Steps

In [ ]:
step_counts = [5, 10, 25, 50, 100, 200]
fig, axes = plt.subplots(1, len(step_counts), figsize=(4 * len(step_counts), 4))

for ax, n_steps in zip(axes, step_counts):
    samples = fm.sample((500, 2), n_steps=n_steps, device=device).cpu()
    ax.scatter(samples[:, 0], samples[:, 1], s=3, alpha=0.5)
    ax.set_title(f'{n_steps} steps')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_aspect('equal')

plt.suptitle('Sample Quality vs Number of Euler Steps', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

1. **Flow matching** replaces the SDE with a deterministic ODE.
2. **Training**: interpolate linearly between noise and data, predict the velocity $v = x_1 - x_0$.
3. **Sampling**: integrate the velocity field using the Euler method from $t=0$ to $t=1$.
4. **Fewer steps**: straight paths mean fewer function evaluations are needed.

### Next: Flow Matching for Tokens

In the next lesson, we adapt this framework to generate text by working in token embedding space.